In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from xgboost import XGBClassifier
from torchvision import models
import torch.nn as nn

# --- CONFIG ---
DATA_DIR = "../data/processed"
MODEL_DIR = "../models"
IMG_DIR = "../data/images"

# 1. Load Data
df = pd.read_csv(f"{DATA_DIR}/final_dataset.csv")
# Create 'address' column for display (if missing, use ID)
if 'address' not in df.columns:
    df['address'] = df['id'].astype(str)

# 2. Load XGBoost (Tabular)
# We assume you just retrain it quickly here to ensure freshness
feature_cols = [c for c in df.columns if c not in ['id', 'target', 'lat', 'lon', 'address', 'filename']]
X = df[feature_cols].values
y = df['target'].values

xgb = XGBClassifier(n_estimators=100, learning_rate=0.05)
xgb.fit(X, y)
df['xgb_prob'] = xgb.predict_proba(X)[:, 1]

# 3. Load CNN (Visual) - Placeholder logic if you want to skip re-running inference
# Ideally, you load the ResNet and loop through tensors like in Notebook 04.
# For the app prototype, if you don't want to wait 20 mins for inference, 
# we can simulate the CNN score or load it if you saved it in Notebook 04.

# OPTION A: If you saved predictions in Notebook 04, load them here.
# OPTION B: If not, we will use a placeholder or re-run inference.
# Let's assume for this step we rely on the XGBoost score for the map, 
# and you can merge CNN scores later if you saved them.
df['cnn_prob'] = df['xgb_prob'] # Placeholder: Replace this with real CNN inference if available!

# 4. Calculate "Risk Factors" (Explanations)
# We categorize homes based on why they are risky
df['risk_factor'] = 'Low Risk'
df.loc[df['fuel_pressure'] > df['fuel_pressure'].median(), 'risk_factor'] = 'High Fuel Load'
df.loc[df['defensible_space_m'] < 5, 'risk_factor'] = 'Zero Defensible Space'
df.loc[(df['fuel_pressure'] > df['fuel_pressure'].median()) & (df['defensible_space_m'] < 5), 'risk_factor'] = 'Critical Vulnerability'

# 5. Save App Data
df.to_csv("../app/app_data.csv", index=False)
print("✅ App data prepared at ../app/app_data.csv")